In [57]:
!pip install gradio -q

In [58]:
!pip install faiss-cpu -q

In [59]:
import pandas as pd
import numpy as np
import pickle
import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer

In [60]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
master_df = pd.read_parquet(
    "/content/drive/MyDrive/PrismHire/master_candidates.parquet"
)

genome_df = pd.read_parquet(
    "/content/drive/MyDrive/PrismHire/models/candidate_genome.parquet"
)

index = faiss.read_index(
    "/content/drive/MyDrive/PrismHire/models/faiss_index.bin"
)

model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cpu"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [62]:
master_df = master_df.merge(
    genome_df,
    on="candidate_id"
)

In [63]:
def explain_candidate(row):

    reasons = []

    # Experience
    if row["years_experience"] >= 8:
        reasons.append("✓ Strong senior-level experience")
    elif row["years_experience"] >= 5:
        reasons.append("✓ Meets required experience level")
    else:
        reasons.append("⚠ Below ideal experience level")

    # Semantic alignment
    if row["semantic_score"] >= 0.60:
        reasons.append("✓ High semantic match with job description")
    elif row["semantic_score"] >= 0.45:
        reasons.append("✓ Moderate semantic alignment")

    # Technical profile
    if row["technical_score"] >= 0.50:
        reasons.append("✓ Strong technical profile")
    elif row["technical_score"] >= 0.30:
        reasons.append("✓ Partial technical overlap")

    # Role fit
    if row["role_score"] >= 0.80:
        reasons.append("✓ Excellent role alignment")
    elif row["role_score"] >= 0.50:
        reasons.append("✓ Good role fit")

    # Recruitability
    if row["recruitability_score"] >= 0.50:
        reasons.append("✓ High recruiter engagement potential")

    # Market demand
    if row["market_score"] >= 0.30:
        reasons.append("✓ Skills are currently in high market demand")

    return reasons

In [64]:
def prismhire_search(job_description):

    # ======================
    # Generate Job Embedding
    # ======================
    job_embedding = model.encode(
        [job_description],
        normalize_embeddings=True
    )

    # ======================
    # Retrieve Top Candidates
    # ======================
    scores, indices = index.search(
        job_embedding.astype("float32"),
        500
    )

    results = master_df.iloc[
        indices[0]
    ].copy()

    # ======================
    # Semantic Similarity
    # ======================
    results["semantic_score"] = scores[0]

    # ======================
    # DNA Scores
    # ======================
    results["technical_score"] = results["technical_dna"]
    results["career_score"] = results["career_dna"]
    results["market_score"] = results["market_dna"]
    results["recruitability_score"] = results["recruitability_dna"]

    # ======================
    # Experience Score
    # ======================
    results["experience_score"] = np.clip(
        results["years_experience"] / 5,
        0,
        1
    )

    # ======================
    # Dynamic Role Matching
    # ======================
    engineering_keywords = [
        "engineer",
        "developer",
        "backend",
        "software",
        "cloud",
        "devops",
        "full stack",
        "java",
        ".net"
    ]

    results["role_score"] = (
        results["current_title"]
        .str.lower()
        .apply(
            lambda x:
            any(
                k in x
                for k in engineering_keywords
            )
        )
        .astype(float)
    )

    # ======================
    # Final Score
    # ======================
    results["final_score"] = (
          0.30 * results["semantic_score"]
        + 0.25 * results["technical_score"]
        + 0.15 * results["career_score"]
        + 0.10 * results["market_score"]
        + 0.10 * results["recruitability_score"]
        + 0.05 * results["experience_score"]
        + 0.05 * results["role_score"]
    )

    # ======================
    # Score Breakdown
    # ======================
    results["score_breakdown"] = (
        "Semantic: "
        + (results["semantic_score"] * 100)
            .round(0)
            .astype(int)
            .astype(str)
        + "% | Technical: "
        + (results["technical_score"] * 100)
            .round(0)
            .astype(int)
            .astype(str)
        + "% | Career: "
        + (results["career_score"] * 100)
            .round(0)
            .astype(int)
            .astype(str)
        + "%"
    )

    # ======================
    # Explanations
    # ======================
    results["explanations"] = results.apply(
        explain_candidate,
        axis=1
    )

    # ======================
    # Ranking
    # ======================
    results = results.sort_values(
        "final_score",
        ascending=False
    ).reset_index(drop=True)

    results.insert(
        0,
        "rank",
        range(1, len(results) + 1)
    )

    return results

In [65]:
def search_and_display(
    job_description,
    top_k
):

    df = prismhire_search(
        job_description
    ).head(top_k)

    df["match_score"] = (
        df["final_score"] * 100
    ).round(1).astype(str) + "%"

    df["explanations"] = (
        df["explanations"]
        .apply(
            lambda x:
            "\n".join(x)
        )
    )

    return df[
        [
            "rank",
            "candidate_id",
            "current_title",
            "years_experience",
            "match_score",
            "score_breakdown",
            "explanations"
        ]
    ]

In [67]:
demo = gr.Interface(
    fn=search_and_display,

    inputs=[
        gr.Textbox(
            lines=10,
            label="Paste Job Description"
        ),

        gr.Slider(
            minimum=5,
            maximum=100,
            value=20,
            step=5,
            label="Top Candidates"
        )
    ],

    outputs=gr.Dataframe(
        wrap=True,
        headers=[
            "Rank",
            "Candidate ID",
            "Role",
            "Experience",
            "Match %",
            "Score Breakdown",
            "Why Recommended"
        ]
    ),

    title="PrismHire",

    description="""
    AI-Powered Candidate Discovery and Ranking Platform

    • Semantic Resume Search
    • Multi-Agent Candidate Ranking
    • Candidate DNA Scoring
    • Explainable AI Recommendations
    """
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b431dc52a188701544.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
